# Financial PhraseBank: Data Preparation and the TF-IDF Baseline

Notebook 2 of 7. It turns the Financial PhraseBank (Malo et al. 2014) into one clean table with a fixed train, validation and test split, then fits the baseline the fine-tuned FinBERT has to beat: TF-IDF features with multinomial Logistic Regression, tuned by cross-validation on the training split only.

Outputs read by later notebooks: `phrasebank.parquet` (every sentence with its label, agreement level and split), `baseline_pred.parquet` (baseline probabilities for the validation and test splits) and `baseline_params.json`. No test metric is computed here; notebook 4 computes them once for every model.

## Setup

Imports, constants and the PhraseBank reader from notebook 1. `SMOKE` shrinks the baseline grid to two points for a quick check run; it is off in this version.

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline

SMOKE = False
SEED = 42
THREADS = 4
INPUT = Path("/kaggle/input")
OUT = Path("/kaggle/working")
LABELS = ["negative", "neutral", "positive"]
SUBSETS = {50: "Sentences_50Agree.txt", 66: "Sentences_66Agree.txt", 75: "Sentences_75Agree.txt", 100: "Sentences_AllAgree.txt"}
START = time.time()


def find_file(name):
    hits = sorted(INPUT.rglob(name))
    if not hits:
        raise FileNotFoundError(name)
    return hits[0]


def elapsed():
    return f"elapsed {(time.time() - START) / 60:.1f} minutes"


def read_text(path):
    raw = path.read_bytes()
    try:
        return raw.decode("utf-8"), "utf-8"
    except UnicodeDecodeError:
        return raw.decode("latin-1"), "latin-1"


def read_subset(level):
    text, encoding = read_text(find_file(SUBSETS[level]))
    rows = []
    for line in text.splitlines():
        if line.strip():
            sentence, _, label = line.strip().rpartition("@")
            rows.append((sentence.strip(), label.strip()))
    return pd.DataFrame(rows, columns=["sentence", "label"]), encoding


print(f"smoke={SMOKE}")
print(f"pandas {pd.__version__}, numpy {np.__version__}, scikit-learn {sklearn.__version__}")

smoke=False
pandas 2.3.3, numpy 2.1.3, scikit-learn 1.6.1


**Conclusion.** The smoke switch is off. The image provides pandas 2.3.3, NumPy 2.1.3 and scikit-learn 1.6.1, so the baseline needs no installs.

## One clean table

The model trains on the 50% agreement file, the full 4.8K sentence set. A sentence repeated with one label is kept once; a sentence whose copies disagree is dropped entirely, since no single label is right for it. Each kept sentence gets the highest agreement level whose file holds it with the same label (notebook 1 confirmed the files are nested), and labels become integers in the fixed order negative, neutral, positive. `sentence_id` is the row number of the cleaned table and links every later prediction back to its sentence.

In [2]:
subsets = {level: read_subset(level)[0] for level in SUBSETS}
pairs = {level: set(zip(frame["sentence"], frame["label"])) for level, frame in subsets.items()}
base = subsets[50]
stats = base.groupby("sentence")["label"].agg(["size", "nunique"])
conflicted = stats.index[stats["nunique"] > 1]
dropped = int(base["sentence"].isin(conflicted).sum())
table = base[~base["sentence"].isin(conflicted)].drop_duplicates("sentence").reset_index(drop=True)
table["agreement"] = [max(level for level in SUBSETS if (s, l) in pairs[level]) for s, l in zip(table["sentence"], table["label"])]
table["label"] = table["label"].map(LABELS.index)
table.insert(0, "sentence_id", np.arange(len(table)))
print(f"lines in the 50% file: {len(base):,}; dropped as conflicting: {dropped}; collapsed as repeats: {len(base) - dropped - len(table)}")
print(f"sentences kept: {len(table):,}")
print(pd.crosstab(table["agreement"], table["label"].map(dict(enumerate(LABELS))), margins=True).to_string())

lines in the 50% file: 4,846; dropped as conflicting: 4; collapsed as repeats: 6
sentences kept: 4,836
label      negative  neutral  positive   All
agreement                                   
50               90      342       195   627
66               94      388       279   761
75              117      755       317  1189
100             303     1386       570  2259
All             604     2871      1361  4836


**Conclusion.** Of the 4,846 lines in the 50% file, 4 are dropped because their sentence carries conflicting labels and 6 are collapsed as repeats, leaving 4,836 sentences, as notebook 1 predicted. The table crosses agreement level with label: 2,259 sentences have full agreement, and the 627 sentences at the 50% level are the most mixed (90 negative, 342 neutral, 195 positive).

## Train, validation and test split

A stratified 80/10/10 split on label crossed with agreement level, seed 42, so every split has the same label mix and the same share of easy and hard sentences. The validation split serves the baseline's sanity check and FinBERT's search; the test split is touched only by notebook 4.

In [3]:
strata = table["label"].astype(str) + "_" + table["agreement"].astype(str)
rest, test = train_test_split(table.index.to_numpy(), test_size=0.1, stratify=strata, random_state=SEED)
train, val = train_test_split(rest, test_size=1 / 9, stratify=strata.loc[rest], random_state=SEED)
table["split"] = "train"
table.loc[val, "split"] = "val"
table.loc[test, "split"] = "test"
order = ["train", "val", "test"]
print(pd.crosstab(table["split"], table["label"].map(dict(enumerate(LABELS)))).reindex(order).to_string())
print(pd.crosstab(table["split"], table["label"].map(dict(enumerate(LABELS))), normalize="index").reindex(order).round(3).to_string())
print(pd.crosstab(table["split"], table["agreement"]).reindex(order).to_string())

label  negative  neutral  positive
split                             
train       484     2296      1088
val          60      287       137
test         60      288       136
label  negative  neutral  positive
split                             
train     0.125    0.594     0.281
val       0.124    0.593     0.283
test      0.124    0.595     0.281
agreement  50   66   75    100
split                         
train      502  609  950  1807
val         63   76  119   226
test        62   76  120   226


**Conclusion.** The split gives 3,868 training, 484 validation and 484 test sentences. Stratifying on label crossed with agreement holds the label shares within 0.2 points across splits (about 12.4% negative, 59.4% neutral, 28.2% positive) and puts 226 full-agreement sentences in each of validation and test. Every sentence sits in exactly one split, since duplicates were collapsed first.

## Tuning the baseline by cross-validation

A baseline is only a fair yardstick if it gets a real search. `GridSearchCV` with five stratified folds on the training split (seed 42) scores accuracy, the same objective FinBERT's search uses, over word n-gram ranges, sublinear term frequency, minimum document frequency, the inverse regularisation strength `C` and class weighting: 144 configurations, 720 fits. The best configuration is refitted on the whole training split.

In [4]:
train_rows = table[table["split"] == "train"]
grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2), (1, 3)],
    "tfidf__sublinear_tf": [False, True],
    "tfidf__min_df": [1, 2],
    "lr__C": [0.1, 0.3, 1.0, 3.0, 10.0, 30.0],
    "lr__class_weight": [None, "balanced"],
}
if SMOKE:
    grid = {"tfidf__ngram_range": [(1, 1)], "lr__C": [1.0, 10.0]}
pipeline = Pipeline([("tfidf", TfidfVectorizer(lowercase=True)), ("lr", LogisticRegression(max_iter=2000))])
search = GridSearchCV(pipeline, grid, scoring="accuracy", cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED), n_jobs=THREADS, refit=True)
search.fit(train_rows["sentence"], train_rows["label"])
results = pd.DataFrame(search.cv_results_)
print(f"configurations searched: {len(results)}, cross-validation fits: {len(results) * 5}")
for _, row in results.sort_values("rank_test_score", kind="stable").head(5).iterrows():
    print(f"rank {row['rank_test_score']}: CV accuracy {row['mean_test_score']:.4f} (sd {row['std_test_score']:.4f}) {row['params']}")
print(f"best: CV accuracy {search.best_score_:.4f} with {search.best_params_}")
print(f"vocabulary of the refitted model: {len(search.best_estimator_.named_steps['tfidf'].vocabulary_):,} terms")
print(elapsed())

configurations searched: 144, cross-validation fits: 720
rank 1: CV accuracy 0.7764 (sd 0.0066) {'lr__C': 10.0, 'lr__class_weight': 'balanced', 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}
rank 2: CV accuracy 0.7753 (sd 0.0056) {'lr__C': 30.0, 'lr__class_weight': 'balanced', 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}
rank 3: CV accuracy 0.7748 (sd 0.0093) {'lr__C': 30.0, 'lr__class_weight': None, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 3), 'tfidf__sublinear_tf': True}
rank 4: CV accuracy 0.7743 (sd 0.0084) {'lr__C': 30.0, 'lr__class_weight': None, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}
rank 5: CV accuracy 0.7740 (sd 0.0108) {'lr__C': 10.0, 'lr__class_weight': None, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2), 'tfidf__sublinear_tf': True}
best: CV accuracy 0.7764 with {'lr__C': 10.0, 'lr__class_weight': 'balanced', 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 2), 'tf

**Conclusion.** All 144 configurations (720 fits) were searched in 4.1 minutes. The best uses word unigrams and bigrams, sublinear term frequency, no minimum document frequency, C = 10 and balanced class weights, at 77.64% cross-validated accuracy (sd 0.66 points across folds). The top five configurations lie within 0.24 points of each other, so the optimum is flat and the choice among them hardly matters. The refitted model has a 49,579-term vocabulary.

## Validation check and the baseline's predictions

Validation accuracy of the refitted baseline shows whether the cross-validation estimate holds on unseen sentences. The cell then writes class probabilities for the validation and test splits; notebook 4 turns the test probabilities into metrics alongside FinBERT's.

In [5]:
assert list(search.classes_) == [0, 1, 2]
val_rows = table[table["split"] == "val"]
test_rows = table[table["split"] == "test"]
val_accuracy = accuracy_score(val_rows["label"], search.predict(val_rows["sentence"]))
print(f"validation accuracy of the refitted baseline: {val_accuracy:.4f} on {len(val_rows)} sentences")
frames = []
for name, rows in (("val", val_rows), ("test", test_rows)):
    probs = search.predict_proba(rows["sentence"])
    frames.append(pd.DataFrame({"sentence_id": rows["sentence_id"].to_numpy(), "split": name, "p_neg": probs[:, 0], "p_neu": probs[:, 1], "p_pos": probs[:, 2]}))
predictions = pd.concat(frames, ignore_index=True)
print(f"probabilities written for {len(val_rows)} validation and {len(test_rows)} test sentences; test metrics are left to notebook 4")

validation accuracy of the refitted baseline: 0.7996 on 484 sentences
probabilities written for 484 validation and 484 test sentences; test metrics are left to notebook 4


**Conclusion.** The refitted baseline reaches 79.96% on the 484 validation sentences, 2.3 points above its cross-validated estimate; with a standard error of about 1.8 points on 484 sentences, that is ordinary sampling variation rather than a sign of leakage. Probabilities for validation and test are written; the test split stays unscored until notebook 4.

## Outputs

Writes the clean table, the baseline's probabilities and its tuned parameters for notebooks 3, 4 and 7.

In [6]:
table.to_parquet(OUT / "phrasebank.parquet", index=False)
predictions.to_parquet(OUT / "baseline_pred.parquet", index=False)
params = {key: list(value) if isinstance(value, tuple) else value for key, value in search.best_params_.items()}
summary = {"params": params, "cv_accuracy": search.best_score_, "val_accuracy": val_accuracy, "configurations": len(results), "scikit_learn": sklearn.__version__}
(OUT / "baseline_params.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
for name in ("phrasebank.parquet", "baseline_pred.parquet", "baseline_params.json"):
    print(f"{name}: {(OUT / name).stat().st_size / 1e3:.0f} kB")

phrasebank.parquet: 411 kB
baseline_pred.parquet: 35 kB
baseline_params.json: 0 kB


**Conclusion.** The clean table (411 kB), the baseline's probabilities (35 kB) and its tuned parameters are written for notebooks 3, 4 and 7.